# 📚 SEG-Y Volume Processing Notebook

This notebook provides a clean, modular, and reusable pipeline to process SEG-Y seismic data for preprocessing tasks like inspection, organization, visualization, and patch extraction. It follows **functional programming principles**, so each operation is encapsulated in a standalone function.

## 🧭 Notebook Outline

1. [Setup & Configuration](#setup)
2. [Read SEG-Y File](#read)
3. [Parse Headers & Metadata](#headers)
4. [Build 3D Seismic Volume](#volume)
5. [Visualize Inline Slices](#visualize)
6. [Extract 3D Patches to HDF5 (Multiprocessing)](#patches)


## 🔧 Setup & Configuration <a name="setup"></a>

Import required libraries and define constants.


In [4]:
import numpy as np
import h5py
from obspy.io.segy.segy import _read_segy
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import matplotlib.colors as mcolors

import os
from PIL import Image
import plotly.graph_objects as go
import h5py

sample_interval_ms = 4  # 4000 µs

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches


## 📥 Read SEG-Y File <a name="read"></a>

Function to load SEG-Y using ObsPy and return a trace stream.


In [5]:
def read_segy_file(path):
    """Read SEG-Y file using ObsPy."""
    return _read_segy(path, headonly=False)


## 🗂 Parse Headers & Metadata <a name="headers"></a>

Extract sorted inline and crossline numbers, and create lookup mappings.


In [6]:
def parse_headers(stream):
    """Extract inline/xline info and mappings."""
    inlines = set()
    xlines = set()
    trace_map = {}
    
    for tr in stream.traces:
        iline = tr.header.for_3d_poststack_data_this_field_is_for_in_line_number
        xline = tr.header.for_3d_poststack_data_this_field_is_for_cross_line_number
        inlines.add(iline)
        xlines.add(xline)
        trace_map[(iline, xline)] = tr.data

    sorted_inlines = sorted(inlines)
    sorted_xlines = sorted(xlines)
    iline_to_idx = {v: i for i, v in enumerate(sorted_inlines)}
    xline_to_idx = {v: i for i, v in enumerate(sorted_xlines)}
    
    return trace_map, sorted_inlines, sorted_xlines, iline_to_idx, xline_to_idx


## 🧱 Build 3D Seismic Volume <a name="volume"></a>

Reorganize SEG-Y trace data into a structured 3D NumPy volume: `[inline, crossline, samples]`.


In [7]:
def build_volume(trace_map, iline_to_idx, xline_to_idx, n_samples):
    """Construct 3D NumPy volume from trace map."""
    volume = np.full((len(iline_to_idx), len(xline_to_idx), n_samples), np.nan, dtype=np.float32)
    for (iline, xline), data in trace_map.items():
        i = iline_to_idx[iline]
        j = xline_to_idx[xline]
        volume[i, j, :] = data
    return volume


In [8]:
def build_mask_volume(masks_path, sorted_inlines, sorted_xlines, n_samples):
    """Build a 3D mask volume from mask images."""
    mask_volume = np.zeros((len(sorted_inlines), len(sorted_xlines), n_samples), dtype=np.uint8)
    
    for iline in sorted_inlines:
        mask_path = os.path.join(masks_path, f"inline_{iline}_mask.png")
        if os.path.exists(mask_path):
            mask_image = Image.open(mask_path).convert("L")  # Convert to grayscale
            mask_array = np.array(mask_image, dtype=np.uint8)
            
            if mask_array.shape[1] != len(sorted_xlines) or mask_array.shape[0] != n_samples:
                raise ValueError(f"Mask dimensions for inline {iline} do not match the seismic volume.")
            
            iline_idx = iline_to_idx[iline]
            mask_volume[iline_idx, :, :] = mask_array.T  # Transpose to match volume orientation
        else:
            raise FileNotFoundError(f"Mask file for inline {iline} not found at {mask_path}.")
    
    return mask_volume

# Build the mask volume


## 🖼 Visualize Inline Slice (Interactive) <a name="visualize"></a>

Interactive viewer using ipywidgets to scroll through inline sections.


In [9]:
def create_inline_viewer(volume, sorted_inlines, sorted_xlines, mask_volume=None, mask_cmap=None):
    """Create an interactive inline viewer with optional mask overlay."""
    
    iline_to_idx = {iline: i for i, iline in enumerate(sorted_inlines)}
    xline_range = sorted_xlines
    if mask_volume is not None and mask_cmap is None:
            unique_labels = np.unique(mask_volume)
            mask_cmap = mcolors.ListedColormap(np.random.rand(len(unique_labels), 3))
    def plot_inline(iline_number):
        iline_index = iline_to_idx[iline_number]
        image = volume[iline_index, :, :].T

        plt.figure(figsize=(12, 6))
        plt.imshow(
            image,
            cmap='gray',
            aspect='auto',
            extent=[min(xline_range), max(xline_range), sample_interval_ms * (image.shape[0] - 1), 0],
            vmin=np.nanmin(volume),
            vmax=np.nanmax(volume)
        )
        plt.title(f"Inline {iline_number}")
        plt.xlabel("Crossline")
        plt.ylabel("Time (ms)")
        plt.colorbar(label="Amplitude")

        if mask_volume is not None:
            mask = mask_volume[iline_index, :, :].T
            plt.imshow(
                mask,
                cmap=mask_cmap,
                alpha=0.5,
                aspect='auto',
                extent=[min(xline_range), max(xline_range), sample_interval_ms * (image.shape[0] - 1), 0]
            )

        plt.show()

    slider = IntSlider(min=min(sorted_inlines), max=max(sorted_inlines),
                       step=1, value=sorted_inlines[len(sorted_inlines) // 2], description="Inline")
    interact(plot_inline, iline_number=slider)



In [10]:
def create_indexed_inline_viewer(volume, sample_interval_ms=4, mask_volume=None, mask_cmap=None):
    """
    Interactive viewer to scroll through inline slices by volume index.
    """

    n_inlines, n_xlines, n_samples = volume.shape

    def plot(iline_index):
        local_cmap = mask_cmap  # avoid reassignment error

        image = volume[iline_index, :, :].T

        plt.figure(figsize=(12, 6))
        plt.imshow(
            image,
            cmap='gray',
            aspect='auto',
            extent=[0, n_xlines, sample_interval_ms * (n_samples - 1), 0],
            vmin=np.nanmin(volume),
            vmax=np.nanmax(volume)
        )
        plt.title(f"Inline Index {iline_index}")
        plt.xlabel("Xline Index")
        plt.ylabel("Time (ms)")
        plt.colorbar(label="Amplitude")

        if mask_volume is not None:
            mask = mask_volume[iline_index, :, :].T
            unique_labels = np.unique(mask)
            if local_cmap is None:
                from matplotlib.colors import ListedColormap
                local_cmap = ListedColormap(np.random.rand(len(unique_labels), 3))
            plt.imshow(
                mask,
                cmap=local_cmap,
                alpha=0.5,
                aspect='auto',
                extent=[0, n_xlines, sample_interval_ms * (n_samples - 1), 0]
            )

        plt.tight_layout()
        plt.show()

    slider = IntSlider(min=0, max=n_inlines - 1, step=1, value=n_inlines // 2, description="Inline Index")
    interact(plot, iline_index=slider)

In [11]:
### TO BE REMOVED
# # 🧱 Build volume
# masks_path = "/Volumes/SSD/data/F3_BIG/masks"
# # 🧠 Parse headers
# stream = read_segy_file("/Volumes/SSD/data/F3_BIG/F3.segy")
# trace_map, sorted_inlines, sorted_xlines, iline_to_idx, xline_to_idx = parse_headers(stream)
# n_samples = next(iter(trace_map.values())).shape[0]
# volume = build_volume(trace_map, iline_to_idx, xline_to_idx, n_samples)
# # mask_volume = build_mask_volume(masks_path, sorted_inlines, sorted_xlines, n_samples)
# # unique_labels = mask_volume.max()
# # mask_volume.shape, volume.shape

# 🖼 Explore inline viewer

# create_inline_viewer(volume, sorted_inlines, sorted_xlines, mask_volume=mask_volume, mask_cmap=mask_cmap)
# create_indexed_inline_viewer(volume, mask_volume=mask_volume, mask_cmap=mask_cmap)

## 📦 Index Patching<a name="index_patches"></a>
Creating 3D patches using only the index

In [ ]:
def generate_patch_centers(volume_shape, patch_size, stride):
    """
    Compute valid patch center indices for a 3D volume using vectorized NumPy ops.

    Parameters:
    - volume_shape (tuple of int): (inlines, xlines, samples)
    - patch_size (tuple of int): Size of the patch in each dimension
    - stride (tuple of int): Stride between patch centers

    Returns:
    - centers (np.ndarray): Array of shape (N, 3) containing all valid patch centers
    """
    assert len(volume_shape) == 3
    assert len(patch_size) == 3
    assert len(stride) == 3

    starts = [np.arange(patch_size[d] // 2,
                        volume_shape[d] - (patch_size[d] - 1) // 2,
                        stride[d])
              for d in range(3)]

    ii, jj, kk = np.meshgrid(starts[0], starts[1], starts[2], indexing='ij')
    centers = np.stack([ii.ravel(), jj.ravel(), kk.ravel()], axis=1)
    return centers

In [ ]:
def extract_patches_batch(volume, centers, patch_size):
    """
    Extract multiple 3D patches from a volume (3D or 4D).

    Parameters:
    - volume (np.ndarray): shape (D1, D2, D3) or (D1, D2, D3, C)
    - centers (np.ndarray): shape (N, 3), each row is (i, j, k)
    - patch_size (tuple): (d1, d2, d3) size of each patch

    Returns:
    - patches (np.ndarray): shape (N, *patch_size) for 3D,
                            or (N, *patch_size, C) for 4D
    """
    assert volume.ndim in [3, 4], "Volume must be 3D or 4D"
    assert centers.ndim == 2 and centers.shape[1] == 3
    assert len(patch_size) == 3

    n_patches = centers.shape[0]
    has_channels = volume.ndim == 4
    half_size = [s // 2 for s in patch_size]

    # Determine output shape
    out_shape = (n_patches, *patch_size)
    if has_channels:
        out_shape += (volume.shape[3],)

    patches = np.empty(out_shape, dtype=volume.dtype)

    for idx, (i, j, k) in tqdm(enumerate(centers)):
        start = (i - half_size[0], j - half_size[1], k - half_size[2])
        end = (start[0] + patch_size[0], start[1] + patch_size[1], start[2] + patch_size[2])

        # Bounds check
        for dim in range(3):
            if start[dim] < 0 or end[dim] > volume.shape[dim]:
                raise ValueError(f"Patch {idx} is out of bounds in dimension {dim}")

        if has_channels:
            patches[idx] = volume[start[0]:end[0], start[1]:end[1], start[2]:end[2], :]
        else:
            patches[idx] = volume[start[0]:end[0], start[1]:end[1], start[2]:end[2]]

    return patches

In [ ]:
def extract_patches_multi_volume(volumes, centers, patch_size, one_hot_configs=None):
    """
    Extract patches from multiple 3D/4D volumes with preallocated memory (high efficiency).

    Parameters:
    - volumes (list of np.ndarray): List of 3D (D1, D2, D3) or 4D (D1, D2, D3, C) arrays
    - centers (np.ndarray): shape (N, 3) of center points
    - patch_size (tuple): (d1, d2, d3) patch size
    - one_hot_configs (list or None): list of integers (num_classes) or None per volume

    Returns:
    - patches (np.ndarray): shape (N, d1, d2, d3, total_channels)
    """
    assert isinstance(volumes, list), "Volumes must be a list"
    assert centers.ndim == 2 and centers.shape[1] == 3
    assert len(patch_size) == 3
    if one_hot_configs is not None:
        assert len(one_hot_configs) == len(volumes)

    n_patches = centers.shape[0]
    half_size = [s // 2 for s in patch_size]

    # --- Calculate total number of channels ---
    channel_counts = []
    for idx, vol in enumerate(volumes):
        if one_hot_configs and one_hot_configs[idx] is not None:
            channel_counts.append(one_hot_configs[idx])  # one-hot: number of classes
        elif vol.ndim == 3:
            channel_counts.append(1)  # raw 3D volume: single channel
        elif vol.ndim == 4:
            channel_counts.append(vol.shape[3])  # already multichannel
        else:
            raise ValueError(f"Unsupported volume dimension at index {idx}")
    
    total_channels = sum(channel_counts)

    # --- Preallocate final patches array ---
    patches = np.empty((n_patches, *patch_size, total_channels), dtype=np.float32)

    # --- Extraction ---
    channel_offset = 0
    for vol_idx, volume in enumerate(volumes):
        one_hot = None if one_hot_configs is None else one_hot_configs[vol_idx]

        if one_hot is not None and volume.ndim == 3:
            # One-hot encode on-the-fly only slices, not full volume expansion
            num_classes = one_hot
            for idx, (i, j, k) in enumerate(centers):
                start = (i - half_size[0], j - half_size[1], k - half_size[2])
                end = (start[0] + patch_size[0], start[1] + patch_size[1], start[2] + patch_size[2])

                for d in range(3):
                    if start[d] < 0 or end[d] > volume.shape[d]:
                        raise ValueError(f"Patch {idx} is out of bounds in dimension {d}")

                patch = volume[start[0]:end[0], start[1]:end[1], start[2]:end[2]]
                one_hot_patch = np.eye(num_classes, dtype=np.float32)[patch]  # shape (d1,d2,d3,C)
                patches[idx, ..., channel_offset:channel_offset+num_classes] = one_hot_patch
        else:
            # Regular 3D or 4D volume
            if volume.ndim == 3:
                volume = volume[..., np.newaxis]  # add singleton channel

            vol_channels = volume.shape[3]
            for idx, (i, j, k) in enumerate(centers):
                start = (i - half_size[0], j - half_size[1], k - half_size[2])
                end = (start[0] + patch_size[0], start[1] + patch_size[1], start[2] + patch_size[2])

                for d in range(3):
                    if start[d] < 0 or end[d] > volume.shape[d]:
                        raise ValueError(f"Patch {idx} is out of bounds in dimension {d}")

                patches[idx, ..., channel_offset:channel_offset+vol_channels] = volume[start[0]:end[0], start[1]:end[1], start[2]:end[2], :]

        channel_offset += channel_counts[vol_idx]

    return patches

## Demo

In [15]:
import data_utils

ModuleNotFoundError: No module named 'data_utils'

In [ ]:
labels_path = "/Volumes/SSD/data/facies_classification_benchmark_data/raw"
labels = np.load(os.path.join(labels_path, "labels_entire_volume.npy"))
processed_seismic = np.load(os.path.join(labels_path, "seismic_entire_volume.npy"))
num_labels = 1+labels.max()

labels.shape, processed_seismic.shape


In [ ]:

colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'cyan', 'magenta', 'brown'][:num_labels]
mask_cmap = mcolors.ListedColormap(colors)
create_indexed_inline_viewer(processed_seismic, mask_volume=labels, mask_cmap=mask_cmap)

In [ ]:
volume_shape = processed_seismic.shape
patch_size = (64,64,64)
stride = (32,32,32)

centers = generate_patch_centers(volume_shape, patch_size, stride)
fixed_inline_index = 256
centers = centers[centers[:, 0] == fixed_inline_index]
print(f"Total patches: {len(centers)}")
print("First 5 centers:", centers[:5])

seis_patches = extract_patches_multi_volume([processed_seismic, labels], centers[:], patch_size, [None, num_labels])
print("Patches size:",seis_patches.shape)


Total patches: 162
First 5 centers: [[256  32  32]
 [256  32  64]
 [256  32  96]
 [256  32 128]
 [256  32 160]]
Patches size: (162, 64, 64, 64, 7)
